# Sliding Window Attention
## 高效处理长序列

<img src="../images/logo.png" width=150>

标准注意力的计算复杂度是O(n²)，当序列长度增加时计算量急剧上升。Sliding Window Attention通过将每个token的注意力限制在固定窗口大小内，将复杂度降为O(n)，可以高效处理超长序列。

Standard attention has O(n²) complexity which explodes with sequence length. Sliding Window Attention limits each token's attention to a fixed window size, reducing complexity to O(n) for efficient long sequence processing.

In [ ]:
class SlidingWindowAttention(nn.Module):
    """
    Sliding Window Attention 实现
    将每个token的注意力限制在窗口大小w内
    
    Sliding Window Attention Implementation
    Limits each token's attention to window size w
    
    修复：
    1. 因果掩码：每个位置只能看到自己和之前的位置
    2. 滑动窗口：每个位置只能看到window_size范围内的token
    """
    def __init__(self, embed_dim, num_heads, window_size=512, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.head_dim = embed_dim // num_heads

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5

    def _create_causal_mask(self, seq_len, device):
        """创建因果掩码：下三角为0（可见），上三角为-inf"""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.masked_fill(mask == 1, float('-inf'))

    def _create_sliding_window_mask(self, seq_len, device):
        """创建滑动窗口掩码：每个位置只能看到window_size范围内的token
        修复：正确的滑动窗口是相对距离，不是绝对位置
        """
        # 创建位置索引
        positions = torch.arange(seq_len, device=device)
        # 计算相对距离矩阵
        diff = positions.unsqueeze(1) - positions.unsqueeze(0)  # (seq_len, seq_len)
        diff = diff.abs()
        # 窗口内的token可见，窗口外的设为-inf
        mask = diff <= self.window_size // 2
        return mask.float().masked_fill(~mask, float('-inf'))

    def forward(self, x, mask=None):
        """x: (batch, seq_len, embed_dim)"""
        B, N, C = x.shape

        # QKV投影 / QKV projection
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(2)

        # 计算注意力分数 / Compute attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale

        # 创建因果+滑动窗口掩码
        causal_mask = self._create_causal_mask(N, x.device)
        window_mask = self._create_sliding_window_mask(N, x.device)

        # 合并掩码：因果和滑动窗口的交集
        combined_mask = causal_mask + window_mask
        combined_mask = combined_mask.masked_fill(combined_mask == 0, float('-inf'))

        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = attn + combined_mask.unsqueeze(0)
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # 应用注意力 / Apply attention
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

# 复杂度对比
## Complexity Comparison

In [ ]:
# 对比标准注意力和滑动窗口注意力的计算量
# Compare computation between standard and sliding window attention

seq_lengths = [128, 256, 512, 1024, 2048, 4096]
window_size = 512

standard_complexity = [n ** 2 for n in seq_lengths]
sliding_complexity = [n * window_size for n in seq_lengths]

plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, standard_complexity, 'b-o', label='Standard Attention O(n²)', linewidth=2)
plt.plot(seq_lengths, sliding_complexity, 'r-s', label=f'Sliding Window O(n×w) w={window_size}', linewidth=2)
plt.xlabel('Sequence Length', fontsize=12)
plt.ylabel('Computations', fontsize=12)
plt.title('Attention Complexity Comparison', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.tight_layout()
plt.savefig('../images/attention_complexity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nComplexity comparison:")
print(f"{'Seq Len':<10} {'Standard O(n²)':<20} {'Sliding O(n×w)':<20} {'Speedup':<10}")
print("-" * 60)
for n, sc, wc in zip(seq_lengths, standard_complexity, sliding_complexity):
    print(f"{n:<10} {sc:<20} {wc:<20} {sc/wc:<10.1f}x")

# 注意力模式可视化
## Attention Pattern Visualization

In [ ]:
# 可视化注意力模式 / Visualize attention patterns

def get_sliding_window_mask(seq_len, window_size):
    """
    获取滑动窗口掩码 / Get sliding window mask
    修复：正确实现相对距离的滑动窗口
    每个位置只能看到 window_size//2 范围内的token
    """
    positions = torch.arange(seq_len)
    diff = positions.unsqueeze(1) - positions.unsqueeze(0)  # (seq_len, seq_len)
    diff = diff.abs()
    # 以当前位置为中心，window_size范围内的token可见
    mask = diff <= window_size // 2
    # 结合因果：只能看到自己和之前的位置
    causal = torch.tril(torch.ones(seq_len, seq_len))
    mask = mask & (causal == 1)
    return mask.float()

# 全局+滑动窗口混合
## Hybrid Global + Sliding Window

In [ ]:
class HybridSlidingWindowAttention(nn.Module):
    """
    混合注意力：结合全局注意力和滑动窗口注意力
    - 全局token（如CLS）可以看到所有位置
    - 普通token使用滑动窗口
    
    Hybrid Attention: Global + Sliding Window
    - Global tokens (e.g., CLS) see all positions
    - Regular tokens use sliding window
    """
    def __init__(self, embed_dim, num_heads, window_size=512, num_global_tokens=1, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.num_global_tokens = num_global_tokens
        self.head_dim = embed_dim // num_heads
        
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x):
        """x: (batch, seq_len, embed_dim)"""
        B, N, C = x.shape
        
        # 分离全局token和普通token / Separate global and regular tokens
        global_tokens = x[:, :self.num_global_tokens]  # (B, num_global, C)
        regular_tokens = x[:, self.num_global_tokens:]  # (B, N-num_global, C)
        
        # 对普通token使用滑动窗口 / Use sliding window for regular tokens
        qkv = self.qkv(regular_tokens).reshape(B, N - self.num_global_tokens, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(2)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # 滑动窗口掩码 / Sliding window mask
        seq_len_reg = N - self.num_global_tokens
        window_mask = torch.zeros_like(attn)
        for i in range(seq_len_reg):
            start = max(0, i - self.window_size // 2)
            end = min(seq_len_reg, i + self.window_size // 2 + 1)
            window_mask[:, i, :start] = float('-inf')
            window_mask[:, i, end:] = float('-inf')
        
        attn = attn + window_mask
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        out_reg = (attn @ v).transpose(1, 2).reshape(B, seq_len_reg, C)
        
        # 全局token通过聚合所有普通token的信息
        # Global tokens aggregate information from all regular tokens
        q_global = self.qkv(global_tokens)[:, :, :C]  # Use first num_global tokens' q
        out_global = q_global.squeeze(1)  # Simplified global output
        
        return self.proj(torch.cat([global_tokens, out_reg], dim=1))

# 测试 / Test
hybrid_attn = HybridSlidingWindowAttention(embed_dim=64, num_heads=4, window_size=8, num_global_tokens=2)
x = torch.randn(2, 34, 64)  # 32 regular + 2 global = 34
output = hybrid_attn(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

In [ ]:
# 注意力感受野可视化 / Attention Receptive Field Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Receptive field growth / 感受野增长
ax1 = axes[0]
layers = [1, 2, 3, 4, 5, 6]
window_sizes = [3, 5, 7, 9, 11, 13]
colors = plt.cm.Blues(np.linspace(0.4, 0.9, 6))

for i, (layer, ws) in enumerate(zip(layers, window_sizes)):
    receptive_field = 2 * layer + 1
    ax1.bar(layer, receptive_field, color=colors[i], edgecolor='black')

ax1.set_xlabel('Layer')
ax1.set_ylabel('Receptive Field Size')
ax1.set_title('Receptive Field Growth
(Each layer expands by window size)')
ax1.set_xticks(layers)

# 2. Global vs Local attention / 全局vs局部注意力
ax2 = axes[1]
seq_len = 16
positions = np.arange(seq_len)

# Global tokens (positions 0 and 15) see all
ax2.axhline(y=0.5, color='red', linestyle='--', linewidth=2, label='Global (CLS token)')
ax2.axhline(y=0.3, color='blue', linestyle='--', linewidth=2, label='Sliding Window (local)')

ax2.fill_between(positions, 0.45, 0.55, alpha=0.3, color='red', label='Global coverage')
ax2.fill_between(positions, 0.25, 0.35, alpha=0.3, color='blue', label='Window coverage')

ax2.set_xlabel('Position in Sequence')
ax2.set_ylabel('Attention Weight')
ax2.set_title('Global vs Local Attention
(Global tokens capture full context)')
ax2.legend()

# 3. Multi-resolution attention / 多分辨率注意力
ax3 = axes[2]
layers_info = {
    'Layer 1-2': {'window': 8, 'type': 'local'},
    'Layer 3-4': {'window': 16, 'type': 'local'},
    'Layer 5-6': {'window': 32, 'type': 'hybrid'},
    'Layer 7-8': {'window': 'full', 'type': 'global'}
}

y_pos = np.arange(len(layers_info))
coverage = [8, 16, 32, 64]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

bars = ax3.barh(y_pos, coverage, color=colors)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(list(layers_info.keys()))
ax3.set_xlabel('Context Coverage (tokens)')
ax3.set_title('Multi-Resolution Attention
(Different layers capture different scales)')

for bar, cov in zip(bars, coverage):
    ax3.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{cov} tokens', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../images/attention_receptive_field.png', dpi=150, bbox_inches='tight')
plt.show()

print("Attention receptive field visualization saved!")

# Longformer/BigBird风格实现
## Longformer/BigBird Style Implementation

In [ ]:
class BigBirdStyleAttention(nn.Module):
    """
    BigBird风格注意力的三种模式：
    1. 滑动窗口注意力 (Sliding)
    2. 全局注意力 (Global)
    3. 随机注意力 (Random)
    
    BigBird-style attention with three patterns:
    1. Sliding window attention
    2. Global attention
    3. Random attention
    """
    def __init__(self, embed_dim, num_heads, window_size=512, num_global=2, num_random=3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.num_global = num_global
        self.num_random = num_random
        self.head_dim = embed_dim // num_heads
        
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(2)
        
        # 初始化注意力矩阵 / Initialize attention matrix
        attn = torch.zeros(B, self.num_heads, N, N, device=x.device)
        
        # 1. 滑动窗口注意力 / Sliding window attention
        for i in range(N):
            start = max(0, i - self.window_size // 2)
            end = min(N, i + self.window_size // 2 + 1)
            attn[:, :, i, start:end] = (q[:, :, i:i+1] @ k[:, :, start:end].transpose(-2, -1)) * self.scale
        
        # 2. 全局token可以看到所有位置 / Global tokens see all positions
        global_q = q[:, :, :self.num_global]
        attn[:, :, :self.num_global, :] = (global_q @ k.transpose(-2, -1)) * self.scale
        attn[:, :, :, :self.num_global] = (q[:, :, :] @ k[:, :, :self.num_global].transpose(-2, -1)) * self.scale
        
        # 3. 随机连接 / Random connections
        for _ in range(self.num_random):
            rand_idx = torch.randint(0, N, (B, N), device=x.device)
            for i in range(N):
                attn[:, :, i, rand_idx[i]] = (q[:, :, i:i+1] @ k[:, :, rand_idx[i]].transpose(-2, -1)) * self.scale
        
        attn = F.softmax(attn, dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

# 测试 / Test
bigbird = BigBirdStyleAttention(embed_dim=64, num_heads=4, window_size=8, num_global=2, num_random=2)
x = torch.randn(2, 32, 64)
output = bigbird(x)
print(f"BigBird-style input: {x.shape} -> output: {output.shape}")

# 与标准Transformer的整合
## Integration with Standard Transformer

In [ ]:
class SlidingWindowTransformerBlock(nn.Module):
    """使用滑动窗口注意力的Transformer块"""
    
    def __init__(self, embed_dim, num_heads, window_size, ff_dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = SlidingWindowAttention(embed_dim, num_heads, window_size)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
    
    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# 测试完整模型 / Test complete model
seq_len = 1024
model = nn.Sequential(
    *[SlidingWindowTransformerBlock(64, 4, window_size=128, ff_dim=256) for _ in range(4)]
)

x = torch.randn(2, seq_len, 64)
import time

start = time.time()
output = model(x)
elapsed = time.time() - start

print(f"Sequence length: {seq_len}")
print(f"Output shape: {output.shape}")
print(f"Forward pass time: {elapsed:.4f}s")
print(f"Throughput: {2 * seq_len / elapsed:.0f} tokens/sec")

# 总结

| 方法 | 复杂度 | 适用场景 |
|------|--------|----------|
| 标准Attention | O(n²) | 短序列 |
| Sliding Window | O(n×w) | 长序列，局部依赖 |
| Longformer | O(n×w + n×g) | 文档级理解 |
| BigBird | O(n×w + n×g + n×r) | 最长上下文 |

滑动窗口注意力是处理长序列的基础，配合RoPE等位置编码可以实现高效的long-context模型。

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **Sliding Window Attention** - O(n×w)复杂度的高效注意力
2. **复杂度对比** - 可视化O(n²)与O(n×w)的差异
3. **注意力模式可视化** - 展示不同窗口大小的注意力覆盖
4. **混合注意力** - 全局token + 滑动窗口
5. **BigBird风格** - 滑动窗口 + 全局 + 随机注意力

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **Flash Attention** | IO-aware高效注意力，无需完整O(n²)矩阵存储 | [Flash Attention](https://arxiv.org/abs/2205.14135) |
| **Paged Attention** | vLLM的显存优化技术，分页管理KV Cache | [vLLM](https://arxiv.org/abs/2309.06180) |
| **StreamingLLM** | 处理无限长度序列的技术 | [StreamingLLM](https://arxiv.org/abs/2309.17453) |
| **Longformer** | BERT-long的实际应用 | [Longformer](https://arxiv.org/abs/2004.05150) |
